# Module 05 — MongoDB

**Formation Big Data — ANSD / Data Innovation Lab**

Nous travaillons sur des **actes d'état civil** : naissances, mariages, décès.
Chaque acte est un document arborescent — un noyau structuré, des sous-documents
pour les parents et le déclarant, et un tableau de **mentions marginales** de
longueur variable.

Le code est fourni. Exécutez, lisez le résultat, et n'hésitez pas à modifier les
valeurs pour explorer.

Sommaire : connexion et import · lecture · écriture · agrégations · index.

## 1. Connexion

In [ ]:
import os
from pprint import pprint

from pymongo import MongoClient

UTILISATEUR = os.environ["MONGO_USER"]
MOT_DE_PASSE = os.environ["MONGO_PASSWORD"]
HOTE = os.environ["MONGO_HOTE"]
PORT = os.environ["MONGO_PORT"]

client = MongoClient(f"mongodb://{UTILISATEUR}:{MOT_DE_PASSE}@{HOTE}:{PORT}/")
client.admin.command("ping")

base = client["etat_civil"]
actes = base["actes"]

print("Connexion établie.")
print("Bases existantes :", client.list_database_names())

## 2. Importer les actes

Le fichier est au format JSON Lines : un document par ligne. On l'insère par
paquets, ce qui est bien plus rapide qu'un document à la fois.

In [ ]:
import json
from pathlib import Path

FICHIER = Path("/home/travail/data/actes.jsonl")

if actes.count_documents({}) > 0:
    print(f"Collection déjà remplie : {actes.count_documents({}):,} actes"
          .replace(",", " "))
else:
    paquet, total = [], 0
    with FICHIER.open(encoding="utf-8") as source:
        for ligne in source:
            paquet.append(json.loads(ligne))
            if len(paquet) == 5_000:
                actes.insert_many(paquet)
                total += len(paquet)
                paquet = []
    if paquet:
        actes.insert_many(paquet)
        total += len(paquet)
    print(f"{total:,} actes importés.".replace(",", " "))

## 3. À quoi ressemble un document ?

In [ ]:
pprint(actes.find_one({"type_acte": "naissance"}))

Remarquez trois choses :

- des **sous-documents** : `centre_etat_civil`, `titulaire`, `parents` ;
- un **tableau** `mentions_marginales`, vide ou non ;
- un `_id` ajouté par MongoDB, identifiant unique du document.

Et surtout, tous les documents n'ont pas les mêmes champs.

In [ ]:
# Trois actes de types différents : les structures diffèrent
for type_acte in ["naissance", "mariage", "deces"]:
    document = actes.find_one({"type_acte": type_acte})
    print(f"{type_acte:<12} → {sorted(document.keys())}")
    print()

In [ ]:
# Certains champs n'existent que sur une partie des documents
total = actes.count_documents({})
for champ in ["jugement_suppletif", "observations", "cause_deces",
              "parents.pere", "temoins"]:
    nombre = actes.count_documents({champ: {"$exists": True}})
    print(f"{champ:<22} présent dans {nombre:>6} actes "
          f"({100 * nombre / total:4.1f} %)")

> C'est exactement ce qu'une table relationnelle représente mal. Pour stocker
> ces actes en SQL, il faudrait une table par type, plus une table pour les
> mentions, plus une table pour les témoins — et quatre jointures pour
> reconstituer un seul acte.

## 4. Lire : les requêtes courantes

Une requête MongoDB est un **document** qui décrit ce que l'on cherche.

In [ ]:
# Égalité simple
print("Naissances :", actes.count_documents({"type_acte": "naissance"}))

# Champ imbriqué : on descend avec des points
print("À Dakar    :", actes.count_documents({"centre_etat_civil.region": "Dakar"}))

# Deux conditions : elles se cumulent
print("Naissances à Dakar :", actes.count_documents({
    "type_acte": "naissance",
    "centre_etat_civil.region": "Dakar",
}))

In [ ]:
# Opérateurs de comparaison : $gt $gte $lt $lte $ne $in $nin
requete = {
    "type_acte": "deces",
    "defunt.age_au_deces": {"$gte": 80},
    "centre_etat_civil.region": {"$in": ["Dakar", "Thiès", "Saint-Louis"]},
}
print("Décès de personnes de 80 ans et plus dans trois régions :",
      actes.count_documents(requete))

# Projection : ne retourner que certains champs (1 pour garder, 0 pour exclure)
for document in actes.find(requete, {"numero_acte": 1, "defunt.age_au_deces": 1,
                                     "_id": 0}).limit(3):
    print(document)

In [ ]:
# Interroger un TABLEAU : la condition s'applique à n'importe quel élément
avec_mention = actes.count_documents({"mentions_marginales": {"$ne": []}})
divorces = actes.count_documents({"mentions_marginales.type": "divorce"})

print(f"Actes portant au moins une mention : {avec_mention:,}".replace(",", " "))
print(f"Actes portant une mention de divorce : {divorces:,}".replace(",", " "))

# Combiner plusieurs conditions sur UN MÊME élément du tableau : $elemMatch
recentes = actes.count_documents({
    "mentions_marginales": {
        "$elemMatch": {"type": "mariage", "date": {"$gte": "2015-01-01"}}
    }
})
print(f"Actes avec un mariage inscrit depuis 2015 : {recentes:,}"
      .replace(",", " "))

In [ ]:
# $or, et la taille d'un tableau
requete = {
    "$or": [
        {"mentions_marginales.2": {"$exists": True}},   # au moins 3 mentions
        {"observations": {"$exists": True}},
    ]
}
print("Actes complexes ou annotés :", actes.count_documents(requete))

# Valeurs distinctes d'un champ
print("Régions :", sorted(actes.distinct("centre_etat_civil.region"))[:5], "…")
print("Types d'actes :", actes.distinct("type_acte"))

## 5. Écrire : insertion, mise à jour, suppression

In [ ]:
# Insertion d'un acte
nouvel_acte = {
    "numero_acte": "NAI-DK-2026-9999999",
    "type_acte": "naissance",
    "date_enregistrement": "2026-08-11",
    "centre_etat_civil": {"region": "Dakar", "commune": "Dakar",
                          "code_centre": "CEC001"},
    "titulaire": {"prenom": "Awa", "nom": "Diop", "sexe": "F",
                  "date_naissance": "2026-08-05"},
    "mentions_marginales": [],
    "numerise": True,
}
resultat = actes.insert_one(nouvel_acte)
print("Inséré avec l'identifiant :", resultat.inserted_id)

In [ ]:
# Mise à jour d'un document : $set modifie, $push ajoute à un tableau
actes.update_one(
    {"numero_acte": "NAI-DK-2026-9999999"},
    {
        "$set": {"observations": "Acte saisi pendant la formation"},
        "$push": {"mentions_marginales": {
            "type": "rectification",
            "date": "2026-08-12",
            "motif": "Erreur d'orthographe du prénom",
        }},
    },
)
pprint(actes.find_one({"numero_acte": "NAI-DK-2026-9999999"},
                      {"_id": 0, "observations": 1, "mentions_marginales": 1}))

In [ ]:
# Mise à jour de plusieurs documents à la fois
resultat = actes.update_many(
    {"type_acte": "naissance", "numerise": False,
     "centre_etat_civil.region": "Kédougou"},
    {"$set": {"numerise": True, "date_numerisation": "2026-08-11"}},
)
print(f"{resultat.modified_count} actes marqués comme numérisés.")

In [ ]:
# Suppression
resultat = actes.delete_one({"numero_acte": "NAI-DK-2026-9999999"})
print(f"{resultat.deleted_count} acte supprimé.")
print(f"Total : {actes.count_documents({}):,}".replace(",", " "))

> 👉 Ouvrez <http://localhost:8081> pour voir la collection dans une interface
> web : les documents, leur structure, et le résultat de vos opérations.

## 6. Agréger

Le pipeline d'agrégation enchaîne des **étapes**, chacune transformant le flux
de documents. C'est l'équivalent d'un `GROUP BY` en SQL, en plus souple.

In [ ]:
# $match filtre, $group regroupe, $sort trie — l'ordre compte
resultat = actes.aggregate([
    {"$match": {"type_acte": "naissance"}},
    {"$group": {
        "_id": "$centre_etat_civil.region",
        "nombre": {"$sum": 1},
    }},
    {"$sort": {"nombre": -1}},
    {"$limit": 5},
])

# SQL équivalent :
#   SELECT region, COUNT(*) FROM actes WHERE type_acte = 'naissance'
#   GROUP BY region ORDER BY COUNT(*) DESC LIMIT 5
for ligne in resultat:
    print(f"{ligne['_id']:<14} {ligne['nombre']:>6}")

In [ ]:
# Regrouper sur deux clés, et calculer plusieurs indicateurs
resultat = actes.aggregate([
    {"$match": {"type_acte": "deces", "defunt.age_au_deces": {"$exists": True}}},
    {"$group": {
        "_id": {"region": "$centre_etat_civil.region", "sexe": "$defunt.sexe"},
        "nombre": {"$sum": 1},
        "age_moyen": {"$avg": "$defunt.age_au_deces"},
        "age_max": {"$max": "$defunt.age_au_deces"},
    }},
    {"$sort": {"nombre": -1}},
    {"$limit": 6},
])
for ligne in resultat:
    cle = ligne["_id"]
    print(f"{cle['region']:<14} {cle['sexe']}  n={ligne['nombre']:>5}  "
          f"âge moyen={ligne['age_moyen']:5.1f}  max={ligne['age_max']}")

In [ ]:
# $unwind : déplier un tableau — une ligne par élément.
# Indispensable pour analyser les mentions marginales.
resultat = actes.aggregate([
    {"$unwind": "$mentions_marginales"},
    {"$group": {"_id": "$mentions_marginales.type", "nombre": {"$sum": 1}}},
    {"$sort": {"nombre": -1}},
])
print("Répartition des mentions marginales :")
for ligne in resultat:
    print(f"  {ligne['_id']:<16} {ligne['nombre']:>6}")

In [ ]:
# Un pipeline plus complet : les mentions par année
resultat = actes.aggregate([
    {"$unwind": "$mentions_marginales"},
    {"$project": {
        "type": "$mentions_marginales.type",
        "annee": {"$substr": ["$mentions_marginales.date", 0, 4]},
    }},
    {"$match": {"annee": {"$gte": "2020"}}},
    {"$group": {"_id": {"annee": "$annee", "type": "$type"},
                "nombre": {"$sum": 1}}},
    {"$sort": {"_id.annee": 1, "nombre": -1}},
    {"$limit": 10},
])
for ligne in resultat:
    print(f"{ligne['_id']['annee']}  {ligne['_id']['type']:<14} "
          f"{ligne['nombre']:>5}")

| Opérateur  | Rôle                                     | Exemple                       |
| ---------- | ---------------------------------------- | ----------------------------- |
| `$unwind`  | Déplie un tableau en plusieurs documents | `["A","B"]` → `"A"` + `"B"`   |
| `$project` | Choisit/modifie les champs affichés      | garder `nom`, supprimer `age` |
| `$group`   | Regroupe les documents                   | compter par catégorie         |
| `$match`   | Filtre les documents                     | équivalent d'un `WHERE` SQL   |


In [ ]:
# Le résultat d'une agrégation s'exploite directement en pandas
import pandas as pd

lignes = list(actes.aggregate([
    {"$group": {
        "_id": {"region": "$centre_etat_civil.region", "type": "$type_acte"},
        "nombre": {"$sum": 1},
    }},
]))

# json_normalize aplatit les clés imbriquées en « _id.region », « _id.type »
tableau = pd.json_normalize(lignes).rename(columns={
    "_id.region": "region", "_id.type": "type_acte"})

tableau.pivot(index="region", columns="type_acte", values="nombre")

## 7. Les index

Sans index, MongoDB lit **tous** les documents pour répondre. Mesurons.

In [ ]:
requete = {"centre_etat_civil.region": "Ziguinchor", "type_acte": "naissance"}

plan = actes.find(requete).explain()["executionStats"]
print("SANS index")
print(f"  documents examinés : {plan['totalDocsExamined']:,}".replace(",", " "))
print(f"  documents retournés : {plan['nReturned']:,}".replace(",", " "))
print(f"  durée : {plan['executionTimeMillis']} ms")

In [ ]:
# Création d'un index composé, sur les deux champs de la requête
actes.create_index([("centre_etat_civil.region", 1), ("type_acte", 1)])

plan = actes.find(requete).explain()["executionStats"]
print("AVEC index")
print(f"  documents examinés : {plan['totalDocsExamined']:,}".replace(",", " "))
print(f"  documents retournés : {plan['nReturned']:,}".replace(",", " "))
print(f"  durée : {plan['executionTimeMillis']} ms")

L'écart tient en une phrase : sans index, MongoDB parcourt la collection
entière ; avec index, il va directement aux documents concernés.

Le prix à payer : un index occupe de l'espace et ralentit les écritures, puisqu'il
faut le tenir à jour. On indexe donc les champs **fréquemment interrogés**, pas
tous les champs.

In [ ]:
# Les index existants de la collection
for nom, definition in actes.index_information().items():
    print(f"{nom:<40} {definition['key']}")

## 8. Ce qu'il faut retenir

- Un **document** peut être imbriqué et irrégulier : c'est l'intérêt du modèle,
  et ce qu'une table représente mal.
- On descend dans les sous-documents avec des **points** :
  `centre_etat_civil.region`.
- Les tableaux s'interrogent comme des champs simples ; `$elemMatch` sert quand
  plusieurs conditions doivent porter sur **le même** élément.
- Le **pipeline d'agrégation** enchaîne `$match`, `$unwind`, `$group`, `$sort` —
  l'ordre des étapes détermine la performance : filtrer d'abord, toujours.
- Un **index** transforme un parcours complet en accès direct. `explain()` dit
  ce qui se passe réellement.